# Backpropagation and The Chain Rule

## 1. Educational Objectives
This notebook breaks down the backpropagation algorithm step-by-step. In our Multilayer Perceptron, backpropagation is how the network learns. After making a prediction (the **forward pass**), the network calculates its error (using **Binary Cross-Entropy Loss**). Backpropagation then works backward from the output layer to the input layer, determining how much each weight and bias in the network contributed to that error.

We use **Gradient Descent** to adjust the weights and minimize the error. To find the gradients (derivatives), we apply the **Chain Rule** of calculus.

### Strict "No-Magic" Constraints
As per our requirements, we are building this entirely from scratch. We will ONLY use `numpy` for matrix operations. No automated differentiation libraries (Autograd, TensorFlow, PyTorch, etc.) are allowed.

## 2. The Chain Rule Matrix Math

During the forward pass, a single layer performs two steps:
1. **Linear combination:** $Z = A_{prev} \cdot W + b$
2. **Activation:** $A = \text{activation}(Z)$

To adjust $W$ and $b$, we need to calculate $\frac{\partial E}{\partial W}$ ($dW$) and $\frac{\partial E}{\partial b}$ ($db$). 
Because the error $E$ is at the very end of the network, we must chain derivatives backwards. Assuming we receive the error signal from the layer ahead of us, denoted as $dA_{next}$ (or $\frac{\partial E}{\partial A}$), we calculate four local matrices:

### 1. The Local Error ($dZ$)
#### <span style="color: red;">The first step tells each individual neuron how wrong it was</span>

$$dZ = dA_{next} \odot \text{activation\_derivative}(Z)$$

<b>where</b>

* $dA_{next}$ is the error signal from the next layer 
* $Z$ is the pre-activation output of the current layer (the linear combination before applying the activation function) during the **forward pass**.
* And $\text{activation\_derivative}(Z)$ is the derivative of the activation function with respect to $Z$. 

When we multiply these together, we get $dZ$, which tells us how much each neuron's output contributed to the error.

*Note: $\odot$ is element-wise multiplication (Hadamard product).*
* To **to undestand more in detail** look the following reference: https://app.notion.com/p/jvalenci/GTD-Getting-Things-Done-14f9d52658e08024afb4f480ed354703?p=3799d52658e0808db1f2c82d78b552f1&pm=s

### 2. The Weight Gradient ($dW$)
#### <span style="color: red;">The second step looks at the whole batch to figure out exactly how to modify the ingredients dials ( the weights) that caused the mistake</span>

Next, we determine how the weights contributed to $Z$. Since $Z = A_{prev} W$, the derivative with respect to $W$ involves $A_{prev}$. By the chain rule across a batch of $m$ examples:
$$dW = \frac{1}{m} (A_{prev}^T \cdot dZ)$$

<b>where:</b>
* $A_{prev}^T$ is the transpose of the input activations from the previous layer or we can call it the input features.
* $dZ$ is the local error we just calculated.
* $dW$ is the average gradient of the weights across the batch, which we will use to update the weights during gradient descent.

<details>
<summary><b> The Calculus Breakdown </b></summary>


Here is exactly how this matrix equation maps perfectly to the calculus you already know.

Our goal is to find **$dW$**. In calculus notation, $dW$ is shorthand for $\frac{\partial \text{Loss}}{\partial W}$ (the derivative of the total Error with respect to the Weights).

To find this using the Chain Rule, we need our two pieces: the "Outside" derivative and the "Inside" derivative.

**1. The "Outside" Derivative ($dZ$)**
We already calculated this in the previous step! $dZ$ is just shorthand for $\frac{\partial \text{Loss}}{\partial Z}$. It represents how much the final Error changes based on the output of the neurons.

**2. The "Inside" Derivative ($A_{prev}$)**
This is where your calculus skills shine.
Think back to the forward pass equation for a neuron:


$$Z = A_{prev} \cdot W$$

Let's take the derivative of $Z$ with respect to the variable $W$.

* Treat $A_{prev}$ exactly like a constant number (like a $5$).
* The variable is $W$ (which is technically $W^1$).
* Using the **Power Rule**, the $W$ drops away entirely, leaving just the constant in front!
* Therefore, the derivative of $A_{prev} \cdot W$ is simply **$A_{prev}$**.

**3. Applying the Chain Rule (Multiply them together)**
The Chain Rule says we multiply the outside derivative by the inside derivative:


$$\frac{\partial \text{Loss}}{\partial W} = \text{Inside Derivative} \times \text{Outside Derivative}$$

$$dW = A_{prev} \times dZ$$

### **Why it looks slightly different**

If we were just doing this for one single number, the equation would literally be $dW = A_{prev} \cdot dZ$.

The *only* reason we add the Transpose ($T$) and the averaging factor ($\frac{1}{m}$) is because we are forcing computers to do this exact calculus operation for thousands of examples and thousands of weights all at the exact same time. The core engine driving the math is entirely the Chain Rule!

</details>

*Why the transpose? To align the dimensions between the (# of examples $\times$ # of features) input matrix and the (# of examples $\times$ # of nodes) error matrix $dZ$.*

### 3. The Bias Gradient ($db$)
Since $b$ is added to every example in the batch, its gradient is the sum of $dZ$ across all examples:
$$db = \frac{1}{m} \sum_{i=1}^{m} dZ^{(i)}$$

### 4. The Upstream Error ($dA_{prev}$)
Finally, we must pass the error backwards so the previous layer can calculate its own $dZ$. Since $Z = A_{prev} W$, the derivative with respect to $A_{prev}$ is $W$:
$$dA_{prev} = dZ \cdot W^T$$
*Why the transpose here? Again, matrix algebra dictates the shapes must align to propagate the (# of nodes) errors backwards into the (# of input features) representation.*